# Painel de Análise de Arquitetura - Global X-Ray Independente (Baseline)
Este notebook consolida a avaliação da arquitetura **Solarfall** operando estritamente sobre a **Inércia Radiativa (Família Global X-Ray)** atuando de forma autônoma (End-to-End).

Para garantir a isonomia do estudo e comprovar a superioridade da Topologia Magnética, a Família Global teve seus hiperparâmetros e limiares de decisão recalibrados sob o mesmo rigor metodológico (maximização de precisão via F-Scores conservadores). O intuito é demonstrar empiricamente que, ao ser forçado a reduzir falsos positivos sem o auxílio de dados locais, o modelo radiativo global perde sensibilidade, assemelhando-se a um modelo de persistência.

A avaliação é realizada **exclusivamente no Ciclo Solar 25 (2020-2024)**, nosso conjunto de Teste Cego com Purga Temporal de 24h.

## 1. Setup & Carregamento de Dados

In [ ]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown
import joblib
import warnings

from src.py_src.models import GatekeeperModel, GreatFilterModel, Specialist910Model, SpecialistMXModel, SolarFlarePredictionModel
from sklearn.metrics import classification_report, average_precision_score, matthews_corrcoef, f1_score

warnings.filterwarnings('ignore', category=FutureWarning)

load_dotenv()

# --- Paths ---
SLIDED_PATH = os.getenv("SLIDED_PATH")
BASE_PATH_XRAY = os.getenv('GLOBAL_XRAY_MODELS_PATH')

# --- Load Data ---
print("Carregando dataset global (Raios-X) do Ciclo 25...")
xray_df = pd.read_parquet(os.path.join(SLIDED_PATH, "xray_slided.parquet"))

test_years = [2020, 2021, 2022, 2023, 2024]

In [ ]:
def extract_test_set_global(df, time_col, purge_hours=24):
    """Extrai Teste Cego para Família Global (com purga temporal obrigatória)."""
    df = df.sort_values(time_col).reset_index(drop=True).copy()
    df['year'] = df[time_col].dt.year
    df['is_test'] = df['year'].isin(test_years)
    df['block_change'] = df['is_test'] != df['is_test'].shift(1)
    df.loc[0, 'block_change'] = False

    drop_indices = set()
    change_indices = df[df['block_change']].index
    purge_td = pd.Timedelta(hours=purge_hours)

    for idx in change_indices:
        t_trans = df.loc[idx, time_col]
        to_drop = df[(df[time_col] >= t_trans - purge_td) & (df[time_col] < t_trans + purge_td)].index
        drop_indices.update(to_drop)

    df_purged = df.drop(index=list(drop_indices)).copy()
    return df_purged[df_purged['is_test']].copy().reset_index(drop=True)

In [ ]:
test_xray = extract_test_set_global(xray_df, 'time')

target_class = 'target_class_in_24h'
target_flux = 'target_flux_in_24h'

X_test_xray = test_xray.drop(columns=[target_class, target_flux, 'time', 'run_id', 'year', 'is_test', 'block_change'], errors='ignore')

print(f"Família Global Autônoma: {len(X_test_xray)} amostras disponíveis no Teste Cego (Ciclo 25)")

## 2. Carregamento dos Modelos End-to-End

In [ ]:
models_xray = {
    'gk': GatekeeperModel.load(os.path.join(BASE_PATH_XRAY, 'gatekeeper_v1.joblib')),
    'gf': GreatFilterModel.load(os.path.join(BASE_PATH_XRAY, 'great_filter_v1.joblib')),
    's910': Specialist910Model.load(os.path.join(BASE_PATH_XRAY, 'specialist_910_v1.joblib')),
    'smx': SpecialistMXModel.load(os.path.join(BASE_PATH_XRAY, 'specialist_mx_v1.joblib'))
}

print("Limiares restritivos embutidos na cascata radiativa autônoma:")
print(f"GK: {models_xray['gk'].threshold:.4f} | GF: {models_xray['gf'].threshold:.4f} | S910: {models_xray['s910'].threshold:.4f}")

## 3. Inferência em Cascata & Avaliação de Escoamento
Analisaremos como a cascata radiativa estrita tenta controlar o alto volume de falsos positivos.

In [ ]:
def analyze_funnel(step_name, y_raw_before, y_raw_after, target_threshold):
    orig_total = len(y_raw_before)
    surv_total = len(y_raw_after)
    if orig_total == 0: return
    
    red_pct = ((orig_total - surv_total) / orig_total) * 100
    orig_pos = (y_raw_before >= target_threshold).sum()
    orig_neg = (y_raw_before < target_threshold).sum()
    surv_pos = (y_raw_after >= target_threshold).sum()
    surv_neg = (y_raw_after < target_threshold).sum()
    
    noise_reduction = ((orig_neg - surv_neg) / orig_neg * 100) if orig_neg > 0 else 0
    signal_retention = (surv_pos / orig_pos * 100) if orig_pos > 0 else 0
    
    display(Markdown(f"**Funnel Report: {step_name}**"))
    print(f"Volume: {orig_total} -> {surv_total} (-{red_pct:.1f}%)")
    print(f"Ruído Eliminado (< Classe Alvo): {noise_reduction:.1f}%")
    print(f"Sinal Retido (>= Classe Alvo): {signal_retention:.1f}%")
    print("-" * 40)

In [ ]:
# --- 1. Gatekeeper ---
y_true_gk = (test_xray[target_class] >= 3).astype(int)
y_pred_gk = models_xray['gk'].predict(X_test_xray)

mask_gf = (y_pred_gk == 1)
analyze_funnel("Entrada -> Gatekeeper", test_xray[target_class], test_xray[target_class][mask_gf], target_threshold=3)

# --- 2. Great Filter ---
X_gf = X_test_xray[mask_gf]
y_true_gf = (test_xray[target_class][mask_gf] >= 3).astype(int)
y_pred_gf = models_xray['gf'].predict(X_gf)

mask_s910 = (y_pred_gf == 1)
y_raw_s910 = test_xray[target_class][mask_gf][mask_s910]
analyze_funnel("Gatekeeper -> Great Filter", test_xray[target_class][mask_gf], y_raw_s910, target_threshold=4)

# --- 3. Specialist 910 (O Juiz das classes M/X) ---
X_s910 = X_gf[mask_s910]
y_true_s910 = (y_raw_s910 >= 4).astype(int)
y_pred_s910 = models_xray['s910'].predict(X_s910)
y_prob_s910 = models_xray['s910'].predict_proba(X_s910)[:, 1]

mask_smx = (y_pred_s910 == 1)
y_raw_smx = y_raw_s910[mask_smx]
analyze_funnel("Great Filter -> Specialist 910", y_raw_s910, y_raw_smx, target_threshold=5)

# --- 4. Specialist MX (Regressão Final) ---
X_smx = X_s910[mask_smx]
y_true_smx = (y_raw_smx >= 5).astype(int)
y_pred_cont = models_xray['smx'].predict(X_smx)
y_pred_smx = (y_pred_cont >= -4.0).astype(int)

## 4. Métricas do Baseline (Specialist 910 Autônomo - X-Ray)
Abaixo extraímos as métricas da classificação severa (M/X vs C) baseada estritamente em inércia global. A expectativa é que a imposição de rigor matemático (Precision) reduza drasticamente a Sensibilidade (Recall) deste modelo comparado à arquitetura topológica.

In [ ]:
tss_fam1 = SolarFlarePredictionModel.calculate_tss(y_true_s910, y_pred_s910)
hss_fam1 = SolarFlarePredictionModel.calculate_hss(y_true_s910, y_pred_s910)
f1_fam1 = f1_score(y_true_s910, y_pred_s910)
pr_auc_s910 = average_precision_score(y_true_s910, y_prob_s910)

print("Resultados do Specialist 910 (Global X-Ray End-to-End):")
print(f"  TSS:    {tss_fam1:.3f}")
print(f"  HSS:    {hss_fam1:.3f}")
print(f"  F1:     {f1_fam1:.3f}")
print(f"  PR-AUC: {pr_auc_s910:.3f}")

print("\nClassification Report (S910 - X-Ray):")
print(models_xray['s910'].get_classification_report(y_true_s910, y_pred_s910, ['< M', 'M/X']))

## 5. Visuais Científicos para a Comparação Final

In [ ]:
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve

# ---------------------------------------------------------
# 5.1 DIAGRAMA DE ESCOAMENTO (FUNNEL REPORT - XRAY)
# ---------------------------------------------------------
volumes_xray = [
    len(X_test_xray),
    len(X_gf),
    len(X_s910),
    len(X_smx)
]

etapas = ["Entrada (Pool Global)", "Sobreviventes do Gatekeeper", "Sobreviventes do Great Filter", "Sobreviventes do S910"]

fig_funnel = go.Figure(go.Funnel(
    name = 'Família Global X-Ray (Autônoma)',
    y = etapas,
    x = volumes_xray,
    textinfo = "value+percent initial",
    marker = {"color": "#ef553b"}
))

fig_funnel.update_layout(
    title="Escoamento da Cascata Global: Restrição de Falsos Positivos",
    yaxis_title="Estágios de Triagem"
)
fig_funnel.show()

# ---------------------------------------------------------
# 5.2 HEATMAP DE ERROS DO SPECIALIST 910 (O Falso Cético)
# ---------------------------------------------------------
err_s910 = models_xray['s910'].analyze_error_distribution(y_true_s910, y_pred_s910, test_xray[target_flux][mask_gf][mask_s910])

plt.figure(figsize=(8, 6))
matriz_erros = err_s910[['FN (Miss)', 'FP (False Alarm)']].astype(float)

sns.heatmap(matriz_erros, annot=True, fmt="g", cmap="Reds", cbar=False,
            annot_kws={"size": 12, "weight": "bold"}, linewidths=.5)
plt.title("Distribuição Física de Erros (Specialist 910 Global Autônomo)", fontsize=14, pad=10)
plt.ylabel("Classe Solar Relevante")
plt.xlabel("Tipo de Erro")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 5.3 CURVA DE PRECISÃO-RECALL (PR-AUC) DA REDE GLOBAL
# ---------------------------------------------------------
precision_I, recall_I, _ = precision_recall_curve(y_true_s910, y_prob_s910)

plt.figure(figsize=(9, 6))
plt.plot(recall_I, precision_I, label=f'Global X-Ray End-to-End - AUC: {pr_auc_s910:.3f}', color='#ef553b', linewidth=2.5)

plt.title('Curva Precisão-Recall Autônoma: Previsão Severa M/X (Baseline)', fontsize=15, pad=15)
plt.xlabel('Recall (Sensibilidade)', fontsize=12)
plt.ylabel('Precisão (VPP)', fontsize=12)
plt.legend(loc="upper right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 5.4 FEATURE IMPORTANCE DO SPECIALIST 910 (X-Ray)
# ---------------------------------------------------------
print("\n--- A INÉRCIA DO ALERTA M/X ---")
importance_df = models_xray['s910'].get_feature_importance().head(10)
importance_df_plot = importance_df.sort_values(by='importance_gain', ascending=True)

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=importance_df_plot,
    x='importance_gain',
    y='feature',
    palette="rocket"
)

for p in ax.patches:
    ax.annotate(f"{p.get_width():.3f}",
                (p.get_width(), p.get_y() + p.get_height() / 2.),
                ha='left', va='center',
                xytext=(5, 0), textcoords='offset points',
                fontsize=10, fontweight='bold')

plt.title('Importância das Variáveis Globais (Specialist 910)', fontsize=15, pad=15)
plt.xlabel('Normalized Information Gain', fontsize=12)
plt.ylabel('Parâmetros Radiativos', fontsize=12)
plt.tight_layout()
plt.show()
